In [ ]:
import cv2
import os
import numpy as np
import pandas as pd
import shutil
from sklearn.metrics import roc_auc_score

import sys
sys.path.append('.')
from config import PROJECT_DIR, RESOLUTIONS, DATASET_ZIP, DATASET_DIR, MODELS, LABELS, MAPS_ZIP, MAPS_DIR
from utils import get_stratification

In [ ]:
os.system(f'unzip -q {DATASET_ZIP} -d {PROJECT_DIR}')
os.system(f'unzip -q {MAPS_ZIP} -d {PROJECT_DIR}')

In [ ]:
IMG_256_DIR = f'{DATASET_DIR}/images_256'
IMG_1024_DIR = f'{DATASET_DIR}/images_1024'
THRESHOLD = 10

## Functions

In [ ]:

def count_zero_padding_borders(img, threshold=THRESHOLD):
    """
    Counts black rows/columns from each border.
    Args:
        img: Grayscale image as a 2D numpy array.
        threshold: Pixel intensity threshold to consider a row/column as black. Permitted values grayscale: 0-255.
    Returns:
        top, bot, left, right: Number of black rows/columns from each border.
    """

    top = 0
    for i in range(img.shape[0]):
        if img[i, :].mean() < threshold:
            top += 1
        else:
            break

    bot = 0
    for i in range(img.shape[0]-1, -1, -1):
        if img[i, :].mean() < threshold:
            bot += 1
        else:
            break

    left = 0
    for j in range(img.shape[1]):
        if img[:, j].mean() < threshold:
            left += 1
        else:
            break

    right = 0
    for j in range(img.shape[1]-1, -1, -1):
        if img[:, j].mean() < threshold:
            right += 1
        else:
            break

    return top, bot, left, right

In [ ]:
def dark_pixels_ratio(img, threshold=THRESHOLD):
    return (img < threshold).sum() / img.size

In [ ]:
# Calculate dark pixels ratio for all images once
dark_ratios_256 = {}
for fname in os.listdir(IMG_256_DIR):
    if not fname.endswith('.png'):
        continue
    image_id = fname.replace('.png', '')
    img = cv2.imread(os.path.join(IMG_256_DIR, fname), cv2.IMREAD_GRAYSCALE)
    if img is None:
        continue
    dark_ratios_256[image_id] = dark_pixels_ratio(img)

dark_ratios_1024 = {}
for fname in os.listdir(IMG_1024_DIR):
    if not fname.endswith('.png'):
        continue
    image_id = fname.replace('.png', '')
    img = cv2.imread(os.path.join(IMG_1024_DIR, fname), cv2.IMREAD_GRAYSCALE)
    if img is None:
        continue
    dark_ratios_1024[image_id] = dark_pixels_ratio(img)

df_dark_256  = pd.DataFrame(list(dark_ratios_256.items()),  columns=['image_id', 'dark_ratio'])
df_dark_1024 = pd.DataFrame(list(dark_ratios_1024.items()), columns=['image_id', 'dark_ratio'])

In [ ]:
# Dark pixels ratio by class for each resolution
dark_pixels_records = []

for res, df_dark, metadata_csv in [
    ('256',  df_dark_256,  f'{DATASET_DIR}/metadata_256.csv'),
    ('1024', df_dark_1024, f'{DATASET_DIR}/metadata_1024.csv')
]:
    print(f'\n=== {res}x{res} ===')
    df_meta = pd.read_csv(metadata_csv)
    df_all = df_meta.drop_duplicates('image_id').merge(df_dark, on='image_id', how='left')

    ids_aneurysm      = set(df_all[df_all['class_id'] == 0]['image_id'])
    ids_cardiomegaly  = set(df_all[df_all['class_id'] == 1]['image_id'])
    ids_healthy       = set(df_all[df_all['class_id'] == 2]['image_id'])
    ids_both          = ids_aneurysm & ids_cardiomegaly
    ids_only_aneurysm = ids_aneurysm - ids_cardiomegaly
    ids_only_cardio   = ids_cardiomegaly - ids_aneurysm

    classes = {
        'Healthy':           ids_healthy,
        'Only aneurysm':     ids_only_aneurysm,
        'Only cardiomegaly': ids_only_cardio,
        'Both':              ids_both
    }

    for name, ids in classes.items():
        df_r = df_dark[df_dark['image_id'].isin(ids)]
        mean = df_r['dark_ratio'].mean()
        std  = df_r['dark_ratio'].std()
        print(f'{name} (n={len(ids)}): mean={mean:.4f} | std={std:.4f}')
        dark_pixels_records.append({
            'resolution': res,
            'class': name,
            'n': len(ids),
            'mean_dark_ratio': round(mean, 4),
            'std_dark_ratio': round(std, 4)
        })

df_dark_pixels = pd.DataFrame(dark_pixels_records)
df_dark_pixels.to_csv(f'{PROJECT_DIR}/dark_pixels_analysis.csv', index=False)
print('\nSaved: dark_pixels_analysis.csv')

In [ ]:
# Zero padding ratio by class for each resolution
results = []

for res, img_dir, metadata_csv in [
    ('256',  IMG_256_DIR,  f'{DATASET_DIR}/metadata_256.csv'),
    ('1024', IMG_1024_DIR, f'{DATASET_DIR}/metadata_1024.csv')
]:
    print(f'\nProcessing {res}x{res}...')
    df_meta = pd.read_csv(metadata_csv)
    files = os.listdir(img_dir)

    ratios = {}
    for f in files:
        img = cv2.imread(os.path.join(img_dir, f), cv2.IMREAD_GRAYSCALE)
        if img is None:
            continue
        image_id = f.replace('.png', '')
        top, bot, left, right = count_zero_padding_borders(img)
        total_px = img.shape[0] * img.shape[1]
        padding_px = (top + bot) * img.shape[1] + (left + right) * img.shape[0]
        ratios[image_id] = padding_px / total_px
        results.append({
            'image_id': image_id,
            'resolution': res,
            'top': top,
            'bottom': bot,
            'left': left,
            'right': right,
            'padding_ratio': padding_px / total_px
        })
    print(f'  {len(files)} images processed.')

    df_padding = pd.DataFrame(list(ratios.items()), columns=['image_id', 'padding_ratio'])
    df_all = df_meta.drop_duplicates('image_id').merge(df_padding, on='image_id', how='left')

    ids_aneurysm      = set(df_all[df_all['class_id'] == 0]['image_id'])
    ids_cardiomegaly  = set(df_all[df_all['class_id'] == 1]['image_id'])
    ids_healthy       = set(df_all[df_all['class_id'] == 2]['image_id'])
    ids_both          = ids_aneurysm & ids_cardiomegaly
    ids_only_aneurysm = ids_aneurysm - ids_cardiomegaly
    ids_only_cardio   = ids_cardiomegaly - ids_aneurysm

    classes = {
        'Healthy':           ids_healthy,
        'Only aneurysm':     ids_only_aneurysm,
        'Only cardiomegaly': ids_only_cardio,
        'Both':              ids_both
    }

    print(f'\n=== {res}x{res} — Zero padding by class ===')
    for name, ids in classes.items():
        df_r = df_padding[df_padding['image_id'].isin(ids)]
        print(f'{name} (n={len(ids)}): mean={df_r["padding_ratio"].mean():.4f} | std={df_r["padding_ratio"].std():.4f}')

df_ratios = pd.DataFrame(results)
for res in RESOLUTIONS:
    df_r = df_ratios[df_ratios['resolution'] == res]
    print(f'\n=== {res}x{res} — Border stats ===')
    for col in ['top', 'bottom', 'left', 'right', 'padding_ratio']:
        print(f'{col}: mean={df_r[col].mean():.3f}, max={df_r[col].max():.1f}')

df_ratios.to_csv(f'{PROJECT_DIR}/zero_padding_analysis.csv', index=False)
print('\nSaved.')

In [ ]:
# Dark pixels ratio by model, label and prediction type
df_preds = pd.read_csv(f'{PROJECT_DIR}/predictions_by_image.csv')
df_preds['type_aneurysm']     = df_preds.apply(lambda r: get_stratification(r, 'aneurysm'), axis=1)
df_preds['type_cardiomegaly'] = df_preds.apply(lambda r: get_stratification(r, 'cardiomegaly'), axis=1)

print('\nMean dark pixels ratio by prediction type:\n')
for label in LABELS:
    col_type = f'type_{label}'
    print(f'── {label.upper()} ──')
    for model in MODELS:
        res = model.split('_')[-1]
        df_dark = df_dark_256 if res == '256' else df_dark_1024
        df_m = df_preds[df_preds['model'] == model].merge(df_dark, on='image_id', how='left')
        summary = df_m.groupby(col_type)['dark_ratio'].mean().round(4)
        print(f'  {model}: {summary.to_dict()}')
    print()

In [ ]:
# Optimal dark pixels threshold as verdict classifier
print("Optimal dark pixels threshold as verdict classifier:\n")
for label in LABELS:
    print(f'── {label.upper()} ──')
    for model in MODELS:
        res = model.split('_')[-1]
        df_dark = df_dark_256 if res == '256' else df_dark_1024
        df_m = df_preds[df_preds['model'] == model].merge(df_dark, on='image_id', how='left').dropna(subset=['dark_ratio'])

        X = df_m['dark_ratio'].values
        y = df_m[f'pred_{label}'].values

        thresholds = np.linspace(X.min(), X.max(), 1000)
        best_acc = 0
        best_threshold = 0
        for t in thresholds:
            pred = (X >= t).astype(int)
            acc = (pred == y).mean()
            if acc > best_acc:
                best_acc = acc
                best_threshold = t

        auc = roc_auc_score(y, X)
        print(f'  {model}: threshold={best_threshold:.4f} | acc={best_acc:.3f} | AUC={auc:.3f}')
    print()

## Restore

In [ ]:
print(f'Removing {DATASET_DIR}...')
shutil.rmtree(DATASET_DIR)
print('... completed.')